# OLMo‑1B  with Hugging Face

This notebook loads allenai/OLMo-1B locally from huggingface and runs generation on Rubin data prompts.

In [43]:
!pip install transformers torch safetensors 


python(92113) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [44]:
!pip install ai2-olmo

python(92114) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [45]:
from hf_olmo import OLMoForCausalLM, OLMoTokenizerFast
import torch
from transformers import GenerationMixin  # Needed for text generation


In [46]:
tokenizer = OLMoTokenizerFast.from_pretrained("allenai/OLMo-1B")
olmo = OLMoForCausalLM.from_pretrained("allenai/OLMo-1B")

# Dynamically extend model to support generate()
class OLMoWithGenerate(GenerationMixin, OLMoForCausalLM):
    pass

olmo.__class__ = OLMoWithGenerate

# Move to GPU if available
device = "cuda" if torch.cuda.is_available() else "cpu"
olmo = olmo.to(device)


In [47]:
def generate_text(prompt, max_new_tokens=1024, temperature=0.7, top_k=50, top_p=0.95):
    inputs = tokenizer([prompt], return_tensors="pt", return_token_type_ids=False)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    output = olmo.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=temperature,
        top_k=top_k,
        top_p=top_p
    )

    return tokenizer.batch_decode(output, skip_special_tokens=True)[0]


In [ ]:

rubin_prompt = (
    "The Rubin Observatory is collecting massive sky survey data. "
    "What are the major challenges in using this dataset to classify galaxies, "
    "considering aspects like data volume, noise, labeling, and model explainability?"
)

response = generate_text(rubin_prompt)
print("Model Response:\n", response)


In [ ]:
rubin_prompt = (
    "Hi"
    "I am following this tutorial:  The LSST Science Pipelines — LSST Science Pipelines  and I\'ve ran the first step “single_frame” task a few times. Each time it runs it produces different results: if I go through all calexps in the output collection ( butler.registry.queryDatasets(\"calexp\", collections=collection) ), and look at their sky coverage (calexp width, height and WCS mapping), and then find the total coverage of the whole collection (max and min ra, dec coordinates), I get different results each time it runs. And I am starting it like this (verbatim what is in the tutorial): "
    " pipetask run -b $RC2_SUBSET_DIR/SMALL_HSC/butler.yaml \
                -p $RC2_SUBSET_DIR/pipelines/DRP.yaml#singleFrame \
                -i HSC/RC2/defaults \
                -o u/$USER/single_frame \
                --register-dataset-types"
    
    "What could be the explanation for this behavior? "
    "Thanks, "
    "Petar"
)

response = generate_text(rubin_prompt)
print("Model Response:\n", response)


Model Response:
 HiI am following this tutorial:  The LSST Science Pipelines — LSST Science Pipelines  and I've ran the first step “single_frame” task a few times. Each time it runs it produces different results: if I go through all calexps in the output collection ( butler.registry.queryDatasets("calexp", collections=collection) ), and look at their sky coverage (calexp width, height and WCS mapping), and then find the total coverage of the whole collection (max and min ra, dec coordinates), I get different results each time it runs. And I am starting it like this (verbatim what is in the tutorial):  pipetask run -b $RC2_SUBSET_DIR/SMALL_HSC/butler.yaml                 -p $RC2_SUBSET_DIR/pipelines/DRP.yaml#singleFrame                 -i HSC/RC2/defaults                 -o u/$USER/single_frame                 --register-dataset-typesWhat could be the explanation for this behavior? Thanks, Petar 

I have also run the same task with the same results.


In [ ]:
rubin_prompt = (
    "I have the following C++ class : "
 "class CcdImageList : public std::list  > "
"The Swig object is correctly transmitted to the Python, but I cannot iterate on the list. "
"What should I do (in Swig ? )  to make it iterable ?"
)

response = generate_text(rubin_prompt)
print("Model Response:\n", response)


Model Response:
 I have the following C++ class : class CcdImageList : public std::list  > The Swig object is correctly transmitted to the Python, but I cannot iterate on the list. What should I do (in Swig? )  to make it iterable?
Is it possible to make a cpp class that is not a C++ class?


In [ ]:
rubin_prompt = (
    "Question on how forced photometry will be run on images taken  after  a DIAObject is detected. Again, using  LSE-163 Data Products and Definitions, Juric et al. 2019-07-29.   - section 3.2.1  states : "
 "\“For all DIAObjects overlapping the field of view … forced photometry will be performed on the difference images. Those measurements will be stored as DIAForced- Sources. No alerts will be issued for these DIAForcedSources, but the DIAForcedSource measurements will be included in any future alerts triggered by a new DIASource at that location.\” "
 "I take this to mean that a DIASource which is found with through the  DIAForcedSource photometry with S/N>5 will have an alert issued. "
 "But what if the  DIAForcedSource produces a measurement S/N < 5 ?  Where will this information be stored - the Prompt Products Database ? "
 "If so, then 3 questions "
 
 "on what timescale will it be available, 24hrs ? "
 "For how long will such forced photometry be run ? The whole survey ? "
 "Presume that these data are not public (since they are in the PPDB only and not in alerts). "
 
 "A strong science case for accessing forced photometry after detection of a transient is fast declining transients e.g. kilonovae, NS-WD mergers. A detection followed by a non-detection is often as interesting as the other way round."
)

response = generate_text(rubin_prompt)
print("Model Response:\n", response)


Model Response:
 Question on how forced photometry will be run on images taken  after  a DIAObject is detected. Again, using  LSE-163 Data Products and Definitions, Juric et al. 2019-07-29.   - section 3.2.1  states : \“For all DIAObjects overlapping the field of view … forced photometry will be performed on the difference images. Those measurements will be stored as DIAForced- Sources. No alerts will be issued for these DIAForcedSources, but the DIAForcedSource measurements will be included in any future alerts triggered by a new DIASource at that location.\” I take this to mean that a DIASource which is found with through the  DIAForcedSource photometry with S/N>5 will have an alert issued. But what if the  DIAForcedSource produces a measurement S/N < 5?  Where will this information be stored - the Prompt Products Database? If so, then 3 questions on what timescale will it be available, 24hrs? For how long will such forced photometry be run? The whole survey? Presume that these data 

In [ ]:
rubin_prompt = (
    "Hi there, "
 "Is there some way I find out what butler repos are available to me? Or, more specifically, how can I list all the values for X that I can put into  butler = dafButler.Butler(X, collections='2.2i/runs/DP0.2') . How can I find out, for example, that I  can  put  dp02  in there, but not, say,   dp85 ? "
 "Thanks!" 
 "James"
)

response = generate_text(rubin_prompt)
print("Model Response:\n", response)


Model Response:
 Hi there, Is there some way I find out what butler repos are available to me? Or, more specifically, how can I list all the values for X that I can put into  butler = dafButler.Butler(X, collections='2.2i/runs/DP0.2'). How can I find out, for example, that I  can  put  dp02  in there, but not, say,   dp85? Thanks!James  


In [ ]:
rubin_prompt = (
    "I’m having trouble building FFTW with texinfo installed on my system. Texinfo builds the FFTW documentation. It looks like the FFTW version in the stack is 3.3.3 (released in '12) which has a known issue with texinfo-5. Supposedly FFTW 3.3.4 (released in March '14) fixes this issue. Anyone else run into this? Should we change the default FFTW version to 3.3.4? "
 "I’ve tried just copying a few of the .texi files from  https://github.com/FFTW/fftw3/tree/master/doc  into my LSST directory but this does not seem to work. I’ve tried aliasing ‘makeinfo’ away but that is not working. Any suggestions?"
)

response = generate_text(rubin_prompt)
print("Model Response:\n", response)


Model Response:
 I’m having trouble building FFTW with texinfo installed on my system. Texinfo builds the FFTW documentation. It looks like the FFTW version in the stack is 3.3.3 (released in '12) which has a known issue with texinfo-5. Supposedly FFTW 3.3.4 (released in March '14) fixes this issue. Anyone else run into this? Should we change the default FFTW version to 3.3.4? I’ve tried just copying a few of the.texi files from  https://github.com/FFTW/fftw3/tree/master/doc  into my LSST directory but this does not seem to work. I’ve tried aliasing ‘makeinfo’ away but that is not working. Any suggestions?
